In [6]:
import os
import random
import glob
from PIL import Image
from collections import defaultdict

def create_reid_visual_grid(data_path, output_name='reid_final_grid.jpg'):
    # 1. 设置参数
    target_h, target_w = 128, 64
    inner_gap = 3   # 同一球员图片之间的间隙
    outer_gap = 9  # 不同球员块（3x3格子）之间的间隙
    
    # 2. 获取并归类
    img_paths = glob.glob(os.path.join(data_path, "*.jpg"))
    pid_to_imgs = defaultdict(list)
    for path in img_paths:
        pid = os.path.basename(path).split('_')[0]
        pid_to_imgs[pid].append(path)
    
    # 3. 筛选并采样 9 个球员
    valid_pids = [pid for pid, imgs in pid_to_imgs.items() if len(imgs) >= 4]
    if len(valid_pids) < 9:
        raise ValueError(f"符合条件的球员不足 9 个，当前仅有 {len(valid_pids)} 个")
    selected_pids = random.sample(valid_pids, 9)

    # 4. 计算画布尺寸
    # 一个球员块的宽度 = 4张图 + 3个内部间隙
    player_block_w = 4 * target_w + 3 * inner_gap
    player_block_h = target_h
    
    # 总宽度 = 3个块 + 2个外部间隙
    total_w = 3 * player_block_w + 2 * outer_gap
    total_h = 3 * player_block_h + 2 * outer_gap
    
    canvas = Image.new('RGB', (total_w, total_h), (255, 255, 255))

    # 5. 循环绘制
    for i in range(9):
        pid = selected_pids[i]
        chosen_imgs = random.sample(pid_to_imgs[pid], 4)
        
        # 3x3 矩阵中的行列
        grid_row = i // 3
        grid_col = i % 3
        
        # 计算该球员块的起始左上角坐标
        start_x = grid_col * (player_block_w + outer_gap)
        start_y = grid_row * (player_block_h + outer_gap)
        
        for j, img_path in enumerate(chosen_imgs):
            # 打开、统一尺寸
            img = Image.open(img_path).convert('RGB')
            img = img.resize((target_w, target_h), Image.LANCZOS)
            
            # 计算每张小图在块内的位置
            x_offset = start_x + j * (target_w + inner_gap)
            y_offset = start_y
            
            canvas.paste(img, (x_offset, y_offset))

    # 6. 保存
    canvas.save(output_name, quality=95)
    print(f"成功！一张包含 9 个球员（每人 4 张图）的可视化图片已保存至: {output_name}")
    canvas.show()

if __name__ == "__main__":
    # 请确保路径正确
    dataset_path = "/home/xiaolei/projects/data/ReID/Qiuxiu/bounding_box_train"
    create_reid_visual_grid(dataset_path)

成功！一张包含 9 个球员（每人 4 张图）的可视化图片已保存至: reid_final_grid.jpg
